In [1]:
import sys, platform
print(sys.version, platform.node())

3.12.13 (main, Mar  4 2026, 09:23:07) [GCC 11.4.0] 648afc7b736a


In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
!wget -q https://mind201910small.blob.core.windows.net/release/MINDsmall_train.zip
!wget -q https://mind201910small.blob.core.windows.net/release/MINDsmall_dev.zip
!unzip -q MINDsmall_train.zip -d MINDsmall_train
!unzip -q MINDsmall_dev.zip -d MINDsmall_dev
!ls MINDsmall_train MINDsmall_dev

unzip:  cannot find or open MINDsmall_train.zip, MINDsmall_train.zip.zip or MINDsmall_train.zip.ZIP.
unzip:  cannot find or open MINDsmall_dev.zip, MINDsmall_dev.zip.zip or MINDsmall_dev.zip.ZIP.
ls: cannot access 'MINDsmall_train': No such file or directory
ls: cannot access 'MINDsmall_dev': No such file or directory


In [4]:
!wget https://mind201910small.blob.core.windows.net/release/MINDsmall_train.zip

--2026-10-09 22:44:54--  https://mind201910small.blob.core.windows.net/release/MINDsmall_train.zip
Resolving mind201910small.blob.core.windows.net (mind201910small.blob.core.windows.net)... 20.60.81.165
Connecting to mind201910small.blob.core.windows.net (mind201910small.blob.core.windows.net)|20.60.81.165|:443... connected.
HTTP request sent, awaiting response... 409 Public access is not permitted on this storage account.
2026-10-09 22:44:54 ERROR 409: Public access is not permitted on this storage account..



In [5]:
!wget https://huggingface.co/datasets/Recommenders/MIND/resolve/main/MINDsmall_train.zip
!wget https://huggingface.co/datasets/Recommenders/MIND/resolve/main/MINDsmall_dev.zip
!unzip -q MINDsmall_train.zip -d MINDsmall_train
!unzip -q MINDsmall_dev.zip -d MINDsmall_dev
!ls MINDsmall_train MINDsmall_dev

--2026-10-09 22:46:22--  https://huggingface.co/datasets/Recommenders/MIND/resolve/main/MINDsmall_train.zip
Resolving huggingface.co (huggingface.co)... 3.167.112.25, 3.167.112.45, 3.167.112.38, ...
Connecting to huggingface.co (huggingface.co)|3.167.112.25|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://us.gcp.cdn.hf.co/xet-bridge-us/6940729b11ef29e2e6969ab2/0d6d50cfb534b494a1153694f8a488170b28b0ac1c254363b565aa6b11c7761f?user_id=public&X-Xet-Cas-Uid=public&response-content-type=application%2Fzip&response-content-disposition=inline%3B+filename*%3DUTF-8%27%27MINDsmall_train.zip%3B+filename%3D%22MINDsmall_train.zip%22%3B&xip=M1J0QNW-ymI&Expires=1791589582&Policy=eyJTdGF0ZW1lbnQiOlt7IlJlc291cmNlIjoiaHR0cHM6Ly91cy5nY3AuY2RuLmhmLmNvL3hldC1icmlkZ2UtdXMvNjk0MDcyOWIxMWVmMjllMmU2OTY5YWIyLzBkNmQ1MGNmYjUzNGI0OTRhMTE1MzY5NGY4YTQ4ODE3MGIyOGIwYWMxYzI1NDM2M2I1NjVhYTZiMTFjNzc2MWZcXD91c2VyX2lkPXB1YmxpYyZYLVhldC1DYXMtVWlkPXB1YmxpYyZyZXNwb25zZS1jb250ZW50LXR5cGU9KiZ

In [6]:
!pwd


/content


In [1]:
import pandas as pd

news = pd.read_csv('MINDsmall_train/news.tsv', sep='\t', header=None,
    names=['news_id','category','subcategory','title','abstract',
           'url','title_entities','abstract_entities'])

behaviors = pd.read_csv('MINDsmall_train/behaviors.tsv', sep='\t', header=None,
    names=['impression_id','user_id','time','history','impressions'])

print("Articles:", news.shape[0])
print("Impressions:", behaviors.shape[0])
print("Users:", behaviors['user_id'].nunique())
print("Categories:", news['category'].nunique())
print(news.head(3))
print(behaviors.head(3))

Articles: 51282
Impressions: 156965
Users: 50000
Categories: 17
  news_id   category      subcategory  \
0  N55528  lifestyle  lifestyleroyals   
1  N19639     health       weightloss   
2  N61837       news        newsworld   

                                               title  \
0  The Brands Queen Elizabeth, Prince Charles, an...   
1                      50 Worst Habits For Belly Fat   
2  The Cost of Trump's Aid Freeze in the Trenches...   

                                            abstract  \
0  Shop the notebooks, jackets, and more that the...   
1  These seemingly harmless habits are holding yo...   
2  Lt. Ivan Molchanets peeked over a parapet of s...   

                                             url  \
0  https://assets.msn.com/labs/mind/AAGH0ET.html   
1  https://assets.msn.com/labs/mind/AAB19MK.html   
2  https://assets.msn.com/labs/mind/AAJgNsz.html   

                                      title_entities  \
0  [{"Label": "Prince Philip, Duke of Edinburgh",...   


In [2]:
print(behaviors['impression_id'].head(10))
print("Unique IDs:", behaviors['impression_id'].nunique(), "Rows:", len(behaviors))

0     1
1     2
2     3
3     4
4     5
5     6
6     7
7     8
8     9
9    10
Name: impression_id, dtype: int64
Unique IDs: 156965 Rows: 156965


In [3]:
# One row per (impression, article shown)
rows = behaviors[['impression_id', 'user_id', 'impressions']].copy()
rows['impressions'] = rows['impressions'].str.split(' ')
rows = rows.explode('impressions')

# Split "N55689-1" into article ID and click label
rows[['news_id', 'clicked']] = rows['impressions'].str.split('-', expand=True)
rows['clicked'] = rows['clicked'].astype(int)
rows = rows.drop(columns='impressions')

print(rows.head())
print("Total (user, article) pairs:", len(rows))
print("Overall click rate:", round(rows['clicked'].mean(), 4))
print("Avg articles shown per impression:", round(rows.groupby('impression_id').size().mean(), 1))

   impression_id user_id news_id  clicked
0              1  U13740  N55689        1
0              1  U13740  N35729        0
1              2  U91836  N20678        0
1              2  U91836  N39317        0
1              2  U91836  N58114        0
Total (user, article) pairs: 5843444
Overall click rate: 0.0404
Avg articles shown per impression: 37.2


In [4]:
rows = rows.reset_index(drop=True)

In [5]:
sizes = rows.groupby('impression_id').size()
print(sizes.describe())

count    156965.000000
mean         37.227688
std          38.594148
min           2.000000
25%          10.000000
50%          24.000000
75%          51.000000
max         299.000000
dtype: float64


In [6]:
clicks_per_visit = rows.groupby('impression_id')['clicked'].sum()
print(clicks_per_visit.describe())

count    156965.000000
mean          1.505711
std           1.153102
min           1.000000
25%           1.000000
50%           1.000000
75%           2.000000
max          35.000000
Name: clicked, dtype: float64


In [7]:
print(behaviors[behaviors['user_id'] == 'U13740'][['impression_id', 'time']])


        impression_id                   time
0                   1  11/11/2019 9:05:58 AM
35262           35263   11/9/2019 5:59:43 AM
154836         154837  11/13/2019 3:27:40 PM


In [8]:
u = behaviors[behaviors['user_id'] == 'U13740']
print(u[['impression_id', 'time']])

# Does each user's history stay the same across all their visits?
hist_versions = behaviors.groupby('user_id')['history'].nunique()
print("Users whose history changes across visits:", (hist_versions > 1).sum())

        impression_id                   time
0                   1  11/11/2019 9:05:58 AM
35262           35263   11/9/2019 5:59:43 AM
154836         154837  11/13/2019 3:27:40 PM
Users whose history changes across visits: 0


In [9]:
# 1. How many articles, and how many are missing an abstract?
print("Articles:", len(news))
print("Missing abstracts:", news['abstract'].isna().sum())

# 2. Category breakdown
print(news['category'].value_counts())

# 3. Title and abstract length (in words)
print("Title words:\n", news['title'].str.split().str.len().describe())
print("Abstract words:\n", news['abstract'].fillna('').str.split().str.len().describe())

# 4. Does every article shown in an impression exist in news.tsv?
shown = set(rows['news_id'])
print("Articles shown in impressions:", len(shown))
print("Shown but missing from news.tsv:", len(shown - set(news['news_id'])))


Articles: 51282
Missing abstracts: 2666
category
news             15774
sports           14510
finance           3107
foodanddrink      2551
lifestyle         2479
travel            2350
video             2068
weather           2048
health            1885
autos             1639
tv                 889
music              769
movies             606
entertainment      587
kids                17
middleeast           2
northamerica         1
Name: count, dtype: int64
Title words:
 count    51282.000000
mean        10.754417
std          3.265311
min          1.000000
25%          9.000000
50%         10.000000
75%         13.000000
max         57.000000
Name: title, dtype: float64
Abstract words:
 count    51282.000000
mean        34.293319
std         26.542819
min          0.000000
25%         15.000000
50%         24.000000
75%         62.000000
max        474.000000
Name: abstract, dtype: float64
Articles shown in impressions: 20288
Shown but missing from news.tsv: 0


In [10]:
shown = set(rows['news_id'])
in_history = set(behaviors['history'].dropna().str.split().explode())
all_news = set(news['news_id'])

print("Shown in impressions:", len(shown))
print("In some user's history:", len(in_history))
print("In both:", len(shown & in_history))
print("Only in history:", len(in_history - shown))
print("In neither:", len(all_news - shown - in_history))

Shown in impressions: 20288
In some user's history: 33195
In both: 2201
Only in history: 30994
In neither: 0


In [11]:
# Date range of the train file (should be Nov 9 to Nov 15)
t = pd.to_datetime(behaviors['time'])
print("Dates:", t.min(), "to", t.max())

# Users and empty histories
users = behaviors.drop_duplicates('user_id')
print("Users:", len(users))
print("Users with empty history:", users['history'].isna().sum())
print("History length:\n", users['history'].fillna('').str.split().str.len().describe())

# Which articles appear where
shown = set(rows['news_id'])
in_history = set(behaviors['history'].dropna().str.split().explode())
all_news = set(news['news_id'])
print("Shown in impressions:", len(shown))
print("In some history:", len(in_history))
print("In both:", len(shown & in_history))
print("Only in history:", len(in_history - shown))
print("In neither:", len(all_news - shown - in_history))

Dates: 2019-11-09 00:00:19 to 2019-11-14 23:59:13
Users: 50000
Users with empty history: 892
History length:
 count    50000.000000
mean        18.521160
std         23.900679
min          0.000000
25%          5.000000
50%         11.000000
75%         22.000000
max        558.000000
Name: history, dtype: float64
Shown in impressions: 20288
In some history: 33195
In both: 2201
Only in history: 30994
In neither: 0
